# OdiaGPT: English → Odia Transformer (From Scratch)
**Dataset:** AI4Bharat Samanantar (en-or) — much cleaner than OPUS  
**Architecture:** Encoder-Decoder Transformer  
**GPU:** RTX 3050 6GB optimized settings

## Changes in This Version (300k)

| Setting | Before | Now |
|---------|--------|-----|
| Train size | 80,000 | 300,000 |
| Val size | 1,000 | 2,000 |
| Time per epoch | ~41 min | ~100 min |
| Expected chrF at epoch 20 | 27-30 | 35-45 |

**Important:** Rename your old `odiagpt/` folder to `odiagpt_80k/` before running to keep old checkpoints.

**Training plan:**
1. Run `train_model(start_epoch=0, total_epochs=20)` — first training
2. Check chrF after epoch 10 to see improvement
3. Resume: `train_model(start_epoch=20, total_epochs=40)` if still improving


## 📁 Folder Structure
```
OdiaGPT/
├── OdiaGPT_Samanantar.ipynb  ← this file
├── odiagpt/                  ← model checkpoints saved here
├── tokenizer_en/             ← English BPE tokenizer
└── tokenizer_or/             ← Odia BPE tokenizer
```

## STEP 0: Install Dependencies

In [1]:
!pip install datasets tokenizers sacrebleu tqdm torch

## STEP 1: Imports & Folder Setup

In [2]:
import os
import math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tqdm import tqdm

# Create output directories (skip if already exist)
for folder in ["./odiagpt", "./tokenizer_en", "./tokenizer_or"]:
    os.makedirs(folder, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.4 GB


## STEP 2: Load Samanantar English-Odia Dataset

In [3]:
# Samanantar is much cleaner than OPUS for Indian languages
print("Loading Samanantar English-Odia dataset...")
raw_dataset = load_dataset("ai4bharat/samanantar", "or", split="train")
print(f"Total dataset size: {len(raw_dataset)}")
print("Sample entry:", raw_dataset[0])

# Samanantar uses 'src' (English) and 'tgt' (Odia)
# We split into train / validation
# Increased to 300k train for better accuracy
# Each epoch ~100 min on RTX 3050 (vs 41 min before)
TRAIN_SIZE = 300000
VAL_SIZE = 2000

if len(raw_dataset) >= TRAIN_SIZE + VAL_SIZE:
    raw_train_dataset, rest = random_split(
        raw_dataset, [TRAIN_SIZE, len(raw_dataset) - TRAIN_SIZE]
    )
    raw_val_dataset, _ = random_split(
        rest, [VAL_SIZE, len(rest) - VAL_SIZE]
    )
else:
    # If dataset is smaller, use 90/10 split
    train_size = int(0.9 * len(raw_dataset))
    val_size = len(raw_dataset) - train_size
    raw_train_dataset, raw_val_dataset = random_split(
        raw_dataset, [train_size, val_size]
    )

print(f"Train size: {len(raw_train_dataset)}")
print(f"Validation size: {len(raw_val_dataset)}")

## STEP 3: Train BPE Tokenizers (English & Odia)

In [4]:
# NOTE: Samanantar uses 'src' for English, 'tgt' for Odia
# (OPUS used 'translation'['en'] and 'translation'['or'])

def get_ds_iterator(dataset, key):
    """key: 'src' for English, 'tgt' for Odia in Samanantar"""
    for data in dataset:
        text = data[key].strip()
        if text:  # skip empty lines
            yield text

SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]

# --- English Tokenizer ---
print("Training English tokenizer...")
tokenizer_en = Tokenizer(BPE(unk_token="[UNK]"))
trainer_en = BpeTrainer(
    min_frequency=2,
    vocab_size=30000,
    special_tokens=SPECIAL_TOKENS
)
tokenizer_en.pre_tokenizer = Whitespace()
tokenizer_en.train_from_iterator(
    get_ds_iterator(raw_train_dataset, "src"),
    trainer=trainer_en
)
tokenizer_en.save("./tokenizer_en/tokenizer_en.json")
print(f"English vocab size: {tokenizer_en.get_vocab_size()}")

# --- Odia Tokenizer ---
print("Training Odia tokenizer...")
tokenizer_or = Tokenizer(BPE(unk_token="[UNK]"))
trainer_or = BpeTrainer(
    min_frequency=2,
    vocab_size=30000,
    special_tokens=SPECIAL_TOKENS
)
tokenizer_or.pre_tokenizer = Whitespace()
tokenizer_or.train_from_iterator(
    get_ds_iterator(raw_train_dataset, "tgt"),
    trainer=trainer_or
)
tokenizer_or.save("./tokenizer_or/tokenizer_or.json")
print(f"Odia vocab size: {tokenizer_or.get_vocab_size()}")

# Reload from files
tokenizer_en = Tokenizer.from_file("./tokenizer_en/tokenizer_en.json")
tokenizer_or = Tokenizer.from_file("./tokenizer_or/tokenizer_or.json")
print("Tokenizers saved and reloaded.")

## STEP 4: Find Max Sequence Length

In [6]:
print("Scanning max sequence lengths (this takes a moment)...")
max_seq_len_src = 0
max_seq_len_tgt = 0

for data in tqdm(raw_train_dataset, desc="Scanning"):
    enc = tokenizer_en.encode(data["src"]).ids
    dec = tokenizer_or.encode(data["tgt"]).ids
    max_seq_len_src = max(max_seq_len_src, len(enc))
    max_seq_len_tgt = max(max_seq_len_tgt, len(dec))

print(f"Max source (EN) length: {max_seq_len_src}")
print(f"Max target (OR) length: {max_seq_len_tgt}")

# Cap at 160 for RTX 3050 6GB safety
# Samanantar has cleaner, shorter sentences than OPUS
max_seq_len = min(max(max_seq_len_src, max_seq_len_tgt) + 5, 160)
print(f"Using max_seq_len = {max_seq_len}")

## STEP 5: Dataset Class & DataLoaders

In [7]:
def causal_mask(size):
    """Upper triangular mask for decoder (prevents attending to future tokens)"""
    mask = torch.triu(torch.ones(1, size, size), diagonal=1).type(torch.int)
    return mask == 0


class EncodeDataset(Dataset):
    def __init__(self, raw_dataset, max_seq_len):
        super().__init__()
        self.raw_dataset = raw_dataset
        self.max_seq_len = max_seq_len
        # Cache special token IDs
        self.CLS_ID = tokenizer_or.token_to_id("[CLS]")
        self.SEP_ID = tokenizer_or.token_to_id("[SEP]")
        self.PAD_ID = tokenizer_or.token_to_id("[PAD]")

    def __len__(self):
        return len(self.raw_dataset)

    def __getitem__(self, idx):
        raw = self.raw_dataset[idx]
        # Samanantar: 'src' = English, 'tgt' = Odia
        source_text = raw["src"]
        target_text = raw["tgt"]

        src_ids = tokenizer_en.encode(source_text).ids[:self.max_seq_len - 2]
        tgt_ids = tokenizer_or.encode(target_text).ids[:self.max_seq_len - 1]

        src_pad_len = self.max_seq_len - len(src_ids) - 2
        tgt_pad_len = self.max_seq_len - len(tgt_ids) - 1

        # encoder_input: [CLS] + src + [SEP] + PAD...
        encoder_input = torch.tensor(
            [self.CLS_ID] + src_ids + [self.SEP_ID] + [self.PAD_ID] * src_pad_len,
            dtype=torch.int64
        )
        # decoder_input: [CLS] + tgt + PAD... (no [SEP] at end during training)
        decoder_input = torch.tensor(
            [self.CLS_ID] + tgt_ids + [self.PAD_ID] * tgt_pad_len,
            dtype=torch.int64
        )
        # target_label: tgt + [SEP] + PAD... (shifted right)
        target_label = torch.tensor(
            tgt_ids + [self.SEP_ID] + [self.PAD_ID] * tgt_pad_len,
            dtype=torch.int64
        )

        encoder_mask = (encoder_input != self.PAD_ID).unsqueeze(0).unsqueeze(0).int()
        decoder_mask = (
            (decoder_input != self.PAD_ID).unsqueeze(0).unsqueeze(0).int()
            & causal_mask(self.max_seq_len)
        )

        return {
            "encoder_input": encoder_input,
            "decoder_input": decoder_input,
            "target_label": target_label,
            "encoder_mask": encoder_mask,
            "decoder_mask": decoder_mask,
            "source_text": source_text,
            "target_text": target_text,
        }


train_ds = EncodeDataset(raw_train_dataset, max_seq_len)
val_ds = EncodeDataset(raw_val_dataset, max_seq_len)

# RTX 3050 6GB: batch_size=16 is safe
# If you get OOM errors, reduce to 8
BATCH_SIZE = 16
train_dataloader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_dataloader = DataLoader(val_ds, batch_size=1, shuffle=True, num_workers=0)

print(f"Train batches: {len(train_dataloader)}")
print(f"Val batches:   {len(val_dataloader)}")

## STEP 6: Transformer Architecture (Unchanged)

In [8]:
# ========================
# Embedding & Positional Encoding
# ========================

class EmbeddingLayer(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_seq_len: int, dropout_rate: float):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        pe = torch.zeros(max_seq_len, d_model)
        pos = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)
        pe = pe.unsqueeze(0)  # (1, max_seq_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, : x.shape[1], :].requires_grad_(False)
        return self.dropout(x)

In [9]:
# ========================
# Multi-Head Attention
# ========================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout_rate: float):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.num_heads = num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, q, k, v, mask):
        query = self.W_q(q)
        key   = self.W_k(k)
        value = self.W_v(v)

        B = query.shape[0]
        query = query.view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        key   = key.view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        value = value.view(B, -1, self.num_heads, self.d_k).transpose(1, 2)

        scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        scores = scores.softmax(dim=-1)
        scores = self.dropout(scores)

        out = scores @ value
        out = out.transpose(1, 2).contiguous().view(B, -1, self.num_heads * self.d_k)
        return self.W_o(out)

In [10]:
# ========================
# FeedForward, LayerNorm, AddAndNorm
# ========================

class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout_rate: float):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.layer_1 = nn.Linear(d_model, d_ff)
        self.layer_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.layer_2(self.dropout(torch.relu(self.layer_1(x))))


class LayerNorm(nn.Module):
    def __init__(self, d_model: int = 512, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta  = nn.Parameter(torch.zeros(d_model))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std  = x.std(dim=-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta


class AddAndNorm(nn.Module):
    def __init__(self, d_model: int, dropout_rate: float):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.layer_norm = LayerNorm(d_model)

    def forward(self, x, sub_layer):
        return x + self.dropout(sub_layer(self.layer_norm(x)))

In [11]:
# ========================
# Encoder
# ========================

class EncoderBlock(nn.Module):
    def __init__(self, d_model, multihead_attention, feed_forward, dropout_rate):
        super().__init__()
        self.multihead_attention = multihead_attention
        self.feed_forward = feed_forward
        self.addnorm_1 = AddAndNorm(d_model, dropout_rate)
        self.addnorm_2 = AddAndNorm(d_model, dropout_rate)

    def forward(self, x, mask):
        x = self.addnorm_1(x, lambda x: self.multihead_attention(x, x, x, mask))
        x = self.addnorm_2(x, self.feed_forward)
        return x


class Encoder(nn.Module):
    def __init__(self, d_model, block_list: nn.ModuleList):
        super().__init__()
        self.block_list = block_list
        self.layer_norm = LayerNorm(d_model)

    def forward(self, x, mask):
        for block in self.block_list:
            x = block(x, mask)
        return self.layer_norm(x)

In [12]:
# ========================
# Decoder
# ========================

class DecoderBlock(nn.Module):
    def __init__(self, d_model, masked_attn, cross_attn, feed_forward, dropout_rate):
        super().__init__()
        self.masked_attn = masked_attn
        self.cross_attn  = cross_attn
        self.feed_forward = feed_forward
        self.addnorm_1 = AddAndNorm(d_model, dropout_rate)
        self.addnorm_2 = AddAndNorm(d_model, dropout_rate)
        self.addnorm_3 = AddAndNorm(d_model, dropout_rate)

    def forward(self, x, enc_out, enc_mask, dec_mask):
        x = self.addnorm_1(x, lambda x: self.masked_attn(x, x, x, dec_mask))
        x = self.addnorm_2(x, lambda x: self.cross_attn(x, enc_out, enc_out, enc_mask))
        x = self.addnorm_3(x, self.feed_forward)
        return x


class Decoder(nn.Module):
    def __init__(self, d_model, block_list: nn.ModuleList):
        super().__init__()
        self.block_list = block_list
        self.layer_norm = LayerNorm(d_model)

    def forward(self, x, enc_out, enc_mask, dec_mask):
        for block in self.block_list:
            x = block(x, enc_out, enc_mask, dec_mask)
        return self.layer_norm(x)

In [13]:
# ========================
# Projection Layer
# ========================

class ProjectionLayer(nn.Module):
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        return self.proj(x)


# ========================
# Full Transformer
# ========================

class Transformer(nn.Module):
    def __init__(self, encoder, decoder, src_embed, tgt_embed,
                 src_pos, tgt_pos, projection_layer):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.src_pos = src_pos
        self.tgt_pos = tgt_pos
        self.projection_layer = projection_layer

    def encode(self, src, src_mask):
        x = self.src_embed(src)
        x = self.src_pos(x)
        return self.encoder(x, src_mask)

    def decode(self, enc_out, src_mask, tgt, tgt_mask):
        x = self.tgt_embed(tgt)
        x = self.tgt_pos(x)
        return self.decoder(x, enc_out, src_mask, tgt_mask)

    def project(self, x):
        return self.projection_layer(x)

In [14]:
# ========================
# Build Model
# ========================

def build_model(
    src_vocab_size, tgt_vocab_size, src_seq_len, tgt_seq_len,
    d_model=512, num_blocks=6, num_heads=8,
    dropout_rate=0.1, d_ff=2048
):
    src_embed = EmbeddingLayer(d_model, src_vocab_size)
    tgt_embed = EmbeddingLayer(d_model, tgt_vocab_size)
    src_pos   = PositionalEncoding(d_model, src_seq_len, dropout_rate)
    tgt_pos   = PositionalEncoding(d_model, tgt_seq_len, dropout_rate)

    encoder_blocks = []
    for _ in range(num_blocks):
        encoder_blocks.append(
            EncoderBlock(
                d_model,
                MultiHeadAttention(d_model, num_heads, dropout_rate),
                FeedForward(d_model, d_ff, dropout_rate),
                dropout_rate
            )
        )

    decoder_blocks = []
    for _ in range(num_blocks):
        decoder_blocks.append(
            DecoderBlock(
                d_model,
                MultiHeadAttention(d_model, num_heads, dropout_rate),
                MultiHeadAttention(d_model, num_heads, dropout_rate),
                FeedForward(d_model, d_ff, dropout_rate),
                dropout_rate
            )
        )

    model = Transformer(
        Encoder(d_model, nn.ModuleList(encoder_blocks)),
        Decoder(d_model, nn.ModuleList(decoder_blocks)),
        src_embed, tgt_embed, src_pos, tgt_pos,
        ProjectionLayer(d_model, tgt_vocab_size)
    )

    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)

    return model


# Build model with RTX 3050-safe settings
# d_model=512, 6 blocks, 8 heads (standard Transformer)
model = build_model(
    tokenizer_en.get_vocab_size(),
    tokenizer_or.get_vocab_size(),
    max_seq_len, max_seq_len,
    d_model=512
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(model)

Model parameters: 90,213,680
Transformer(
  (encoder): Encoder(
    (block_list): ModuleList(
      (0-5): 6 x EncoderBlock(
        (multihead_attention): MultiHeadAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (W_q): Linear(in_features=512, out_features=512, bias=False)
          (W_k): Linear(in_features=512, out_features=512, bias=False)
          (W_v): Linear(in_features=512, out_features=512, bias=False)
          (W_o): Linear(in_features=512, out_features=512, bias=False)
        )
        (feed_forward): FeedForward(
          (dropout): Dropout(p=0.1, inplace=False)
          (layer_1): Linear(in_features=512, out_features=2048, bias=True)
          (layer_2): Linear(in_features=2048, out_features=512, bias=True)
        )
        (addnorm_1): AddAndNorm(
          (dropout): Dropout(p=0.1, inplace=False)
          (layer_norm): LayerNorm()
        )
        (addnorm_2): AddAndNorm(
          (dropout): Dropout(p=0.1, inplace=False)
          (layer_

## STEP 7: Training Loop (with Resume Support)

In [17]:
def train_model(start_epoch=0, total_epochs=30):
    """
    Train the model.
    start_epoch=0  → train from scratch
    start_epoch=N  → resume from ./odiagpt/model_N.pt
    total_epochs   → how many epochs in total (not additional)
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, eps=1e-9)

    # Resume from checkpoint
    if start_epoch > 0:
        ckpt_path = f"./odiagpt/model_{start_epoch}.pt"
        checkpoint = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        print(f"Resumed from epoch {start_epoch}")

    loss_fn = nn.CrossEntropyLoss(
        ignore_index=tokenizer_or.token_to_id("[PAD]"),
        label_smoothing=0.1
    ).to(device)

    for epoch in range(start_epoch + 1, total_epochs + 1):
        model.train()
        epoch_loss = 0
        loop = tqdm(train_dataloader, desc=f"Epoch {epoch:03d}")

        for batch in loop:
            optimizer.zero_grad(set_to_none=True)

            enc_in    = batch["encoder_input"].to(device)
            dec_in    = batch["decoder_input"].to(device)
            enc_mask  = batch["encoder_mask"].to(device)
            dec_mask  = batch["decoder_mask"].to(device)
            labels    = batch["target_label"].to(device)

            enc_out    = model.encode(enc_in, enc_mask)
            dec_out    = model.decode(enc_out, enc_mask, dec_in, dec_mask)
            logits     = model.project(dec_out)

            loss = loss_fn(
                logits.view(-1, tokenizer_or.get_vocab_size()),
                labels.view(-1)
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
            optimizer.step()

            epoch_loss += loss.item()
            loop.set_postfix(loss=f"{loss.item():.3f}")

        avg_loss = epoch_loss / len(train_dataloader)
        print(f"\nEpoch {epoch} | Avg Loss: {avg_loss:.4f}")

        # Save checkpoint
        torch.save(
            {"model_state_dict": model.state_dict(), "epoch": epoch},
            f"./odiagpt/model_{epoch}.pt"
        )

    print("\nTraining complete!")

In [18]:
# ▶ Run this to train from scratch for 30 epochs
# Approx time on RTX 3050 6GB: ~25-35 min/epoch
train_model(start_epoch=0, total_epochs=20)

In [ ]:
# ▶ To RESUME training (e.g., continue from epoch 30 to 60):
# train_model(start_epoch=30, total_epochs=60)

## STEP 8: Inference with Beam Search

In [20]:
# Load best checkpoint before inference
BEST_EPOCH = 20 # Change this to whichever epoch you want to test

checkpoint = torch.load(f"./odiagpt/model_{BEST_EPOCH}.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()
print(f"Loaded model from epoch {BEST_EPOCH}")

In [21]:
def clean_odia_text(text):
    """Remove special tokens and common noise from decoded output"""
    banned = ["[CLS]", "[SEP]", "[PAD]", "[UNK]", "[MASK]",
              "Name", "I / O", "unit", "format", "_u"]
    for token in banned:
        text = text.replace(token, "")
    # Clean up double spaces
    while "  " in text:
        text = text.replace("  ", " ")
    return text.strip()


def odiagpt_beam(text, beam_size=3, max_len=None):
    """
    Translate English text to Odia using beam search.
    beam_size: 3 recommended for this model size
    max_len: MUST match training max_seq_len
    """
    if max_len is None:
        max_len = max_seq_len  # use training value automatically

    model.eval()
    with torch.no_grad():

        # ===== ENCODE INPUT =====
        CLS_EN = tokenizer_en.token_to_id("[CLS]")
        SEP_EN = tokenizer_en.token_to_id("[SEP]")
        PAD_EN = tokenizer_en.token_to_id("[PAD]")

        src = tokenizer_en.encode(text).ids
        src = [CLS_EN] + src[:max_len - 2] + [SEP_EN]
        src += [PAD_EN] * (max_len - len(src))
        src = torch.tensor(src, dtype=torch.int64).unsqueeze(0).to(device)
        src_mask = (src != PAD_EN).unsqueeze(1).unsqueeze(2).int()
        encoder_output = model.encode(src, src_mask)

        # ===== BEAM INIT =====
        CLS = tokenizer_or.token_to_id("[CLS]")
        SEP = tokenizer_or.token_to_id("[SEP]")
        PAD = tokenizer_or.token_to_id("[PAD]")

        beams = [(torch.tensor([[CLS]], device=device), 0.0)]
        completed = []

        # ===== BEAM DECODE =====
        for _ in range(max_len):
            new_beams = []
            for seq, score in beams:
                if seq[0, -1].item() == SEP:
                    completed.append((seq, score))
                    continue

                dec_mask = causal_mask(seq.size(1)).to(device)
                out = model.decode(encoder_output, src_mask, seq, dec_mask)
                logits = model.project(out[:, -1])
                log_probs = torch.log_softmax(logits, dim=-1)
                topk_probs, topk_ids = torch.topk(log_probs, beam_size)

                for i in range(beam_size):
                    next_id = topk_ids[0, i].item()
                    next_score = score + topk_probs[0, i].item()
                    new_seq = torch.cat(
                        [seq, torch.tensor([[next_id]], device=device)], dim=1
                    )
                    new_beams.append((new_seq, next_score))

            beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]
            if len(completed) >= beam_size:
                break

        best_seq = completed[0][0] if completed else beams[0][0]

        # ===== DECODE TOKENS =====
        tokens = [t for t in best_seq[0].tolist() if t not in {CLS, SEP, PAD}]
        decoded = tokenizer_or.decode(tokens)
        return clean_odia_text(decoded)


print("Beam search inference ready!")

Beam search inference ready!


In [32]:
# ===== TEST TRANSLATIONS =====
# Use news/formal sentences (Samanantar is news-domain)
test_sentences = [
    "He was born in India.",
    "This incident happened yesterday.",
    "The government announced a new policy.",
    "She is studying in the university.",
    "The village was affected by floods.",
    "odisha is best",
]

for sent in test_sentences:
    translation = odiagpt_beam(sent, beam_size=3)
    print(f"EN: {sent}")
    print(f"OR: {translation}")
    print("-" * 60)

EN: He was born in India.
OR: ସେ ଭାରତରେ ଜନ୍ମ ନେଇଛନ୍ତି ।
------------------------------------------------------------
EN: This incident happened yesterday.
OR: ଏଭଳି ଘଟଣା ଘଟିଛି ।
------------------------------------------------------------
EN: The government announced a new policy.
OR: ସରକାର ନୂଆ ନୀତି ଘୋଷଣା କରିଛନ୍ତି ।
------------------------------------------------------------
EN: She is studying in the university.
OR: ସେ ସାମ୍ବାଦିକତା ବିଶ୍ୱବିଦ୍ୟାଳୟର ପାଠ ପଢ଼ୁଥିଲେ ।
------------------------------------------------------------
EN: The village was affected by floods.
OR: ଫଳରେ ଗାଁରେ ଆତଙ୍କ ଖେଳିଯାଇଛି ।
------------------------------------------------------------
EN: odisha is best
OR: ସେଣ୍ଟ କୁମାରସ୍ୱାମୀ
------------------------------------------------------------


## STEP 9: BLEU Score Evaluation

In [27]:
import sacrebleu

# Use 50 samples from validation set for BLEU
# More samples = more reliable score
EVAL_SIZE = 500

predictions = []
references  = []

print(f"Evaluating on {EVAL_SIZE} validation samples...")

for i, sample in enumerate(tqdm(raw_val_dataset)):
    if i >= EVAL_SIZE:
        break
    en_text = sample["src"]
    or_ref  = sample["tgt"]

    if not en_text.strip() or not or_ref.strip():
        continue

    try:
        pred = odiagpt_beam(en_text, beam_size=3)
        predictions.append(pred)
        references.append(or_ref)
    except Exception as e:
        print(f"Skipped sample {i}: {e}")
        continue

# Compute BLEU
bleu = sacrebleu.corpus_bleu(predictions, [references])
print(f"\n{'='*50}")
print(f"BLEU Score: {bleu.score:.2f}")
print(f"{'='*50}")
print()
print("BLEU interpretation for Odia from scratch:")
print("  < 5   → model is learning language structure")
print("  5-10  → basic alignment working")
print("  10-18 → decent for low-resource from-scratch")
print("  18+   → excellent (rare without pretraining)")

In [28]:
# ===== Show sample predictions vs reference =====
print("\nSample Predictions vs Reference:\n")
for i in range(min(10, len(predictions))):
    print(f"[{i+1}]")
    print(f"  REF  : {references[i]}")
    print(f"  PRED : {predictions[i]}")
    print()


Sample Predictions vs Reference:

[1]
  REF  : ଭାରତ ଏବଂ ଆମେରିକାର ଏହି ସ୍ୱତନ୍ତ୍ର ବନ୍ଧୁତାର ସବୁଠୁ ଗୁରୁତ୍ୱପୂର୍ଣ୍ଣ ମୂଳଦୁଆ ହେଉଛି ଆମ ନାଗରିକମାନଙ୍କ ମଧ୍ୟରେ ସମ୍ପର୍କ ।
  PRED : ଏହି ଅବସରରେ ଆମେରିକା ଏବଂ ଭାରତ ମଧ୍ୟରେ ଥିବା ସ୍ଵତନ୍ତ୍ର ବନ୍ଧୁତା ରହିଛି ।

[2]
  REF  : ନୀଳ ବିପ୍ଳବ ଯୋଜନା ଅଧୀନରେ ଲମ୍ବା ଲାଇନ୍ ଟ୍ରଲର ହିତାଧିକାରୀମାନଙ୍କୁ ସ୍ୱୀକୃତିପତ୍ର ପ୍ରଦାନ କରିବେ ।
  PRED : ଏହି ଯୋଜନାର ଶୁଭାରମ୍ଭ କରି ପ୍ରଧାନମନ୍ତ୍ରୀ ଆବାସ ଯୋଜନାର ଶୁଭାରମ୍ଭ କରିବେ ।

[3]
  REF  : ଏସୀୟ କ୍ରୀଡ଼ା ବାସ୍କେଟ୍‌ବଲ୍‌ରେ ଭାରତକୁ ରୌପ୍ୟ
  PRED : ରାଜ୍ୟଗୋଷ୍ଠୀ କ୍ରୀଡ଼ାର ରୌପ୍ୟ – ରୌପ୍ୟ , ରୌପ୍ୟ ବିଜେତା , ରୌପ୍ୟ

[4]
  REF  : ଏହା ଦ୍ବାରା ବାସ୍ତୁ ଦୋଷ ଦୂର ହୋଇଥାଏ।
  PRED : ଏହି ସ୍ଥାନରେ ହାଲକା ଲାଲ କୋବି ରହିଥାଏ ।

[5]
  REF  : ମାଗିଥିବା ଅଭିଯୋଗ ହୋଇଥିଲା।
  PRED : ଯାହାକୁ ନେଇ ପ୍ରଶ୍ନ ଉଠାଇଛି ।

[6]
  REF  : ସେହି ସମୟରେ ଏହାର ନାମ ରୟାଲ ଇଣ୍ଡିଆନ୍ ଏୟାର ଫୋର୍ସ ଥିଲା ।
  PRED : ତା ’ ପରଠାରୁ ପାକିସ୍ତାନର ଏହି ଏୟାର ସର୍ଜିକାଲ ଷ୍ଟ୍ରାଇକ ଘଟିଛି ।

[7]
  REF  : ମାତ୍ର ବିଜେପି ଓ କଂଗ୍ରେସ ମଧ୍ୟରେ କଡା ଟକ୍କର ହେବ।
  PRED : ତେବେ କଂଗ୍ରେସ ଓ ବିଜେପି ମଧ୍ୟରେ ଲଢ଼େଇ ହେବ ।

[8]
  REF  : ଏହା ସାଧାରଣ ନିଷ୍ପତ୍ତି ନଥିଲା ବନ୍ଧୁଗଣ ରାଜନୀତି ଦ

## STEP 10: Resume Training (Run this cell to continue training)

After checking BLEU, if you want to improve, resume training from where you stopped.

In [ ]:
# ▶ RESUME TRAINING EXAMPLE
# Uncomment and change the numbers:

# train_model(start_epoch=30, total_epochs=60)
# After 60 epochs: load model_60.pt and run BLEU again

# BEST_EPOCH = 60
# checkpoint = torch.load(f"./odiagpt/model_{BEST_EPOCH}.pt", map_location=device)
# model.load_state_dict(checkpoint["model_state_dict"])
# model.eval()

In [33]:
# Resume training from epoch 10 → 50
# Approx time: ~41 min/epoch × 40 epochs = ~27 hours total
# You can stop at any point (e.g., run to 30 first, check BLEU, then continue)

train_model(start_epoch=10, total_epochs=20)

C:\Users\Bithal Kumar Sahoo\AppData\Local\Temp\ipykernel_11476\2139655258.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_locatio

Resumed from epoch 10


Epoch 011: 100%|██████████| 5000/5000 [42:04<00:00,  1.98it/s, loss=3.378]



Epoch 11 | Avg Loss: 3.5042


Epoch 012: 100%|██████████| 5000/5000 [41:40<00:00,  2.00it/s, loss=3.079]


...


Epoch 18 | Avg Loss: 2.2855


Epoch 019: 100%|██████████| 5000/5000 [1:41:57<00:00,  1.22s/it, loss=2.333]      



Epoch 19 | Avg Loss: 2.2188


Epoch 020: 100%|██████████| 5000/5000 [1:01:10<00:00,  1.36it/s, loss=2.389]  



Epoch 20 | Avg Loss: 2.1635

Training complete!


In [35]:
# Load your new best checkpoint
BEST_EPOCH = 20  # change to whatever epoch you stopped at

checkpoint = torch.load(f"./odiagpt/model_{BEST_EPOCH}.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded epoch {BEST_EPOCH}")

# Run BLEU again
predictions = []
references  = []

for i, sample in enumerate(tqdm(raw_val_dataset)):
    if i >= 500:
        break
    try:
        pred = odiagpt_beam(sample["src"], beam_size=3)
        predictions.append(pred)
        references.append(sample["tgt"])
    except:
        continue

bleu = sacrebleu.corpus_bleu(predictions, [references])
print(f"BLEU Score at epoch {BEST_EPOCH}: {bleu.score:.2f}")

In [37]:
import sacrebleu

# Use 'char' tokenizer — better for Odia script
bleu_char = sacrebleu.corpus_bleu(
    predictions, 
    [references], 
    tokenize='char'   # character-level, fairer for Indic scripts
)
print(f"BLEU (char tokenizer): {bleu_char.score:.2f}")

# Also try chrF which is better for morphologically rich languages like Odia
chrf = sacrebleu.corpus_chrf(predictions, [references])
print(f"chrF Score: {chrf.score:.2f}")

In [38]:
BEST_EPOCH = 20

In [39]:
checkpoint = torch.load(f"./odiagpt/model_{BEST_EPOCH}.pt", map_location=device)

C:\Users\Bithal Kumar Sahoo\AppData\Local\Temp\ipykernel_11476\1734946628.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(f"./odiagpt/model_{BEST

In [40]:
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [41]:
model.eval()

Transformer(
  (encoder): Encoder(
    (block_list): ModuleList(
      (0-5): 6 x EncoderBlock(
        (multihead_attention): MultiHeadAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (W_q): Linear(in_features=512, out_features=512, bias=False)
          (W_k): Linear(in_features=512, out_features=512, bias=False)
          (W_v): Linear(in_features=512, out_features=512, bias=False)
          (W_o): Linear(in_features=512, out_features=512, bias=False)
        )
        (feed_forward): FeedForward(
          (dropout): Dropout(p=0.1, inplace=False)
          (layer_1): Linear(in_features=512, out_features=2048, bias=True)
          (layer_2): Linear(in_features=2048, out_features=512, bias=True)
        )
        (addnorm_1): AddAndNorm(
          (dropout): Dropout(p=0.1, inplace=False)
          (layer_norm): LayerNorm()
        )
        (addnorm_2): AddAndNorm(
          (dropout): Dropout(p=0.1, inplace=False)
          (layer_norm): LayerNorm()
        )


In [42]:
predictions = []
references  = []

In [43]:
for i, sample in enumerate(tqdm(raw_val_dataset)):
    if i >= 500:
        break

 50%|█████     | 500/1000 [00:00<00:00, 17592.97it/s]


In [45]:
pred = odiagpt_beam(sample["src"], beam_size=3)
predictions.append(pred)
references.append(sample["tgt"])

In [46]:
bleu = sacrebleu.corpus_bleu(predictions, [references])

In [47]:
# ===== MANUAL TEST WITH ACCURACY CHECK =====

test_cases = [
    # (English input, correct Odia reference)
    ("He was born in India.", "ସେ ଭାରତରେ ଜନ୍ମ ନେଇଥିଲେ ।"),
    ("The government announced a new policy.", "ସରକାର ଏକ ନୂଆ ନୀତି ଘୋଷଣା କଲେ ।"),
    ("This incident happened yesterday.", "ଏହି ଘଟଣା ଗତକାଲି ଘଟିଥିଲା ।"),
    ("She is studying in the university.", "ସେ ବିଶ୍ୱବିଦ୍ୟାଳୟରେ ପଢ଼ୁଛନ୍ତି ।"),
    ("The village was affected by floods.", "ଗ୍ରାମଟି ବନ୍ୟାରେ କ୍ଷତିଗ୍ରସ୍ତ ହୋଇଥିଲା ।"),
    ("India and America signed an agreement.", "ଭାରତ ଏବଂ ଆମେରିକା ଏକ ଚୁକ୍ତି ସ୍ୱାକ୍ଷର କଲେ ।"),
    ("The police arrested the accused.", "ପୋଲିସ ଅଭିଯୁକ୍ତକୁ ଗିରଫ କଲା ।"),
    ("Children should go to school.", "ପିଲାମାନେ ବିଦ୍ୟାଳୟ ଯିବା ଉଚିତ ।"),
]

test_preds = []
test_refs  = []

print("=" * 70)
print(f"{'ENGLISH':<35} | ODIA TRANSLATION")
print("=" * 70)

for en, ref in test_cases:
    pred = odiagpt_beam(en, beam_size=3)
    test_preds.append(pred)
    test_refs.append(ref)
    print(f"\nEN  : {en}")
    print(f"REF : {ref}")
    print(f"PRED: {pred}")

# Compute chrF for this test set
import sacrebleu
chrf_test = sacrebleu.corpus_chrf(test_preds, [test_refs])
bleu_test  = sacrebleu.corpus_bleu(test_preds, [test_refs], tokenize='char')

print("\n" + "=" * 70)
print(f"chrF Score  : {chrf_test.score:.2f}  (character F-score, best for Odia)")
print(f"BLEU (char) : {bleu_test.score:.2f}  (character BLEU)")
print("=" * 70)
print("\nInterpretation:")
print("  chrF 25-35  → decent for epoch 20, from scratch")
print("  chrF 35-45  → good, keep training to epoch 50")
print("  chrF 45+    → excellent for Odia")

In [48]:
import sacrebleu

def test_single(english, reference_odia):
    """Test one English sentence and score it against a reference."""
    pred = odiagpt_beam(english, beam_size=3)
    
    # Sentence-level chrF
    chrf = sacrebleu.sentence_chrf(pred, [reference_odia])
    
    print("=" * 60)
    print(f"EN  : {english}")
    print(f"REF : {reference_odia}")
    print(f"PRED: {pred}")
    print(f"chrF: {chrf.score:.2f}")
    print("=" * 60)
    
    return pred, chrf.score

# ---- Test your own sentences here ----
test_single(
    "The election results were announced today.",
    "ଆଜି ନିର୍ବାଚନ ଫଳ ଘୋଷଣା କରାଗଲା ।"
)

test_single(
    "Farmers are demanding better prices for their crops.",
    "କୃଷକମାନେ ସେମାନଙ୍କ ଫସଲ ପାଇଁ ଭଲ ମୂଲ୍ୟ ଦାବି କରୁଛନ୍ତି ।"
)

test_single(
    "The hospital was inaugurated by the Chief Minister.",
    "ମୁଖ୍ୟମନ୍ତ୍ରୀ ଡାକ୍ତରଖାନାର ଉଦ୍ଘାଟନ କଲେ ।"
)

In [52]:
test_single("The minister visited the flood-affected area.", "")
test_single("Two people were injured in the accident.", "")

EN  : The minister visited the flood-affected area.
REF : 
PRED: ବନ୍ୟା ପ୍ରଭାବିତ ଅଂଚଳରେ ମୁଖ୍ୟମନ୍ତ୍ରୀ ପ୍ରଭାବିତ ହୋଇଛନ୍ତି ।
chrF: 0.00
EN  : Two people were injured in the accident.
REF : 
PRED: ଦୁର୍ଘଟଣାରେ ଦୁଇ ଜଣ ଆହତ ହୋଇଛନ୍ତି ।
chrF: 0.00


('ଦୁର୍ଘଟଣାରେ ଦୁଇ ଜଣ ଆହତ ହୋଇଛନ୍ତି ।', 0.0)

In [53]:
pred = odiagpt_beam("The minister visited the flood-affected area.", beam_size=3)
print(pred)

ବନ୍ୟା ପ୍ରଭାବିତ ଅଂଚଳରେ ମୁଖ୍ୟମନ୍ତ୍ରୀ ପ୍ରଭାବିତ ହୋଇଛନ୍ତି ।


In [60]:
pred = odiagpt_beam("The police arrested two people.", beam_size=3)
print(pred)

pred = odiagpt_beam("Heavy rain fell in Odisha.", beam_size=3)
print(pred)

pred = odiagpt_beam("The court gave its verdict today.", beam_size=3)
print(pred)


ପୋଲିସ ଦୁଇ ଜଣଙ୍କୁ ଗିରଫ କରିଛି ।
ଏହାର ପ୍ରଭାବରେ ଓଡିଶାରେ ପ୍ରବଳ ବର୍ଷା ଲାଗି ରହିଛି ।
କୋର୍ଟ ଆଜି ଏହାର ଶୁଣାଣି କରି ସାରିଛନ୍ତି ।


In [77]:
import sacrebleu
from tqdm import tqdm

# ===== ACCURACY EVALUATION =====

# Test sentences with correct Odia references
test_cases = [
    ("The police arrested two people.", "ପୋଲିସ ଦୁଇ ଜଣଙ୍କୁ ଗିରଫ କଲା ।"),
    ("Heavy rain fell in Odisha.", "ଓଡ଼ିଶାରେ ପ୍ରବଳ ବର୍ଷା ହୋଇଛି ।"),
    ("The court gave its verdict today.", "କୋର୍ଟ ଆଜି ରାୟ ଦେଇଛି ।"),
    ("The government announced a new scheme.", "ସରକାର ଏକ ନୂଆ ଯୋଜନା ଘୋଷଣା କଲେ ।"),
    ("Two people were killed in the accident.", "ଦୁର୍ଘଟଣାରେ ଦୁଇ ଜଣ ମୃତ୍ୟୁ ବରଣ କଲେ ।"),
    ("The election results were declared.", "ନିର୍ବାଚନ ଫଳ ଘୋଷଣା ହୋଇଛି ।"),
    ("India won the cricket match.", "ଭାରତ କ୍ରିକେଟ ମ୍ୟାଚ ଜିତିଛି ।"),
    ("The school was closed due to rain.", "ବର୍ଷା କାରଣରୁ ବିଦ୍ୟାଳୟ ବନ୍ଦ ହୋଇଗଲା ।"),
    ("The minister resigned from his post.", "ମନ୍ତ୍ରୀ ନିଜ ପଦରୁ ଇସ୍ତଫା ଦେଲେ ।"),
    ("Farmers protested on the highway.", "କୃଷକମାନେ ରାଜପଥରେ ଧରଣା ଦେଲେ ।"),
    
]

predictions = []
references  = []

print(f"{'#':<3} {'ENGLISH':<45} {'chrF'}")
print("=" * 70)

for i, (en, ref) in enumerate(test_cases):
    pred = odiagpt_beam(en, beam_size=3)
    predictions.append(pred)
    references.append(ref)
    
    # per-sentence chrF
    score = sacrebleu.sentence_chrf(pred, [ref]).score
    print(f"{i+1:<3} {en:<45} {score:.1f}")

# Overall scores
print("=" * 70)
chrf_overall = sacrebleu.corpus_chrf(predictions, [references])
bleu_overall = sacrebleu.corpus_bleu(predictions, [references], tokenize='char')

print(f"\nOverall chrF  : {chrf_overall.score:.2f}")
print(f"Overall BLEU  : {bleu_overall.score:.2f}")
print(f"Avg per sentence chrF: {sum(sacrebleu.sentence_chrf(p,[r]).score for p,r in zip(predictions,references))/len(predictions):.2f}")

print("\n--- Predictions vs References ---")
for i, (pred, ref) in enumerate(zip(predictions, references)):
    print(f"\n[{i+1}] REF : {ref}")
    print(f"    PRED: {pred}")

In [78]:
# Pull random sentences directly from validation set
# These were NEVER seen during training
import random

random.seed(42)
val_samples = random.sample(list(raw_val_dataset), 20)

predictions = []
references  = []

print("UNSEEN validation sentences:\n")
for sample in val_samples:
    en  = sample["src"]
    ref = sample["tgt"]
    pred = odiagpt_beam(en, beam_size=3)
    
    predictions.append(pred)
    references.append(ref)
    
    score = sacrebleu.sentence_chrf(pred, [ref]).score
    print(f"EN  : {en}")
    print(f"REF : {ref}")
    print(f"PRED: {pred}")
    print(f"chrF: {score:.1f}")
    print("-" * 60)

# Real overall score
chrf = sacrebleu.corpus_chrf(predictions, [references])
print(f"\nReal chrF on unseen data: {chrf.score:.2f}")

In [79]:
new_sentences = [
    "The train was delayed due to heavy fog.",
    "The patient was admitted to the hospital.",
    "Three students won gold medals in the competition.",
    "The bridge was constructed over the river.",
    "Odisha government launched a new welfare scheme.",
    "The accused was sentenced to five years in jail.",
    "Farmers received compensation for crop damage.",
    "The fire broke out in a factory last night.",
]

print("NEW SENTENCE TRANSLATIONS")
print("=" * 65)

for en in new_sentences:
    pred = odiagpt_beam(en, beam_size=3)
    print(f"EN  : {en}")
    print(f"OR  : {pred}")
    print("-" * 65)

NEW SENTENCE TRANSLATIONS
EN  : The train was delayed due to heavy fog.
OR  : ପ୍ରବଳ ବର୍ଷା ଯୋଗୁଁ ଟ୍ରେନ ବିଳମ୍ବରେ ପଡ଼ିଥିଲା ।
-----------------------------------------------------------------
EN  : The patient was admitted to the hospital.
OR  : ରୋଗୀ କୁ ମେଡିକାଲରେ ଭର୍ତ୍ତି କରିଥିଲେ ।
-----------------------------------------------------------------
EN  : Three students won gold medals in the competition.
OR  : ତିନିଟି ପ୍ରତିଯୋଗିତାରେ ସଫଳତା ଲାଭ କରିଥିବା ଛାତ୍ରଛାତ୍ରୀଙ୍କୁ ପୁରସ୍କାର ମିଳିଛି ।
-----------------------------------------------------------------
EN  : The bridge was constructed over the river.
OR  : ନଦୀର ସେତୁ ଭଳି ସେତୁ ନିର୍ମାଣ ହୋଇଥିଲା ।
-----------------------------------------------------------------
EN  : Odisha government launched a new welfare scheme.
OR  : ଓଡ଼ିଶା ସରକାର ନୂଆ ଯୋଜନା ଆରମ୍ଭ କରିଛନ୍ତି ।
-----------------------------------------------------------------
EN  : The accused was sentenced to five years in jail.
OR  : ଅଭିଯୁକ୍ତ ପାଞ୍ଚ ବର୍ଷ ହେଲା ଜେଲ ଦଣ୍ଡାଦେଶ ।
------------------------------